# FixtureIQ — Full Data Exploration: All Sources (2022-23 to 2024-25)

## Objectives
1. Inventory every data source available in the project
2. Check missing values, data quality, schema compatibility
3. Understand name matching (SofaScore vs FBref vs Injury)
4. Assess cross-source merge feasibility
5. Build a unified view of what data is usable

## Seasons Covered: 2022-23, 2023-24, 2024-25

## Data Sources
- **SofaScore Dynamic** (`Data/<season>/sofascore_dynamic/`) — per-match player stats + workload features
- **FBref per-match** (`Data/<season>/fbref/`) — per-match player reports for UCL teams
- **SofaScore season aggregates** (`Data/<season>/sofascore/`) — season-level player stats
- **Injuries** (`Data/<season>/injuries/`) — injury records for all PL teams
- **Club Elo** (`Data/clubelo_understat/fixtureiq_elo_master.csv`) — team strength ratings
- **Understat** (`Data/clubelo_understat/fixtureiq_understat_master.csv`) — xG match data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, re
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

BASE = Path('..')
print(f'Project root: {BASE.resolve()}')
print(f'Data exists: {(BASE / "Data").exists()}')

---
## 1. SofaScore Dynamic Data (the primary workload dataset)

These are per-match player records with pre-engineered workload features: `rest_days`, `acwr_ratio`, `high_congestion_flag`, `min_last_7d`, etc.

In [ ]:
def explore_sofascore_dynamic(season_code):
    """Load and explore the SofaScore Dynamic clean dataset for a given season."""
    path = BASE / 'Data' / season_code / 'sofascore_dynamic' / 'fixtureiq_dynamic_analytics_clean.csv'
    master_path = BASE / 'Data' / season_code / 'sofascore_dynamic' / 'fixtureiq_dynamic_master.csv'
    
    if not path.exists():
        print(f'[WARN] {season_code}: clean file not found')
        return None, None
    
    df = pd.read_csv(path)
    df_master = pd.read_csv(master_path) if master_path.exists() else None
    
    print(f'=== {season_code} SofaScore Dynamic ===')
    print(f'Clean: {df.shape[0]:,} rows x {df.shape[1]} cols')
    if df_master is not None:
        print(f'Master: {df_master.shape[0]:,} rows x {df_master.shape[1]} cols')
    
    # Date range
    date_col = 'match_date_str' if 'match_date_str' in df.columns else 'date'
    print(f'Date range: {df[date_col].min()} to {df[date_col].max()}')
    
    # Missing values
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f'\nMissing values:')
        for c, v in nulls.items():
            print(f'  {c}: {v:>6} ({v/df.shape[0]*100:5.1f}%)')
    else:
        print('\nMissing values: (none)')
    
    # Teams & players
    print(f'\nUnique teams: {df["teamName"].nunique()}')
    print(f'Unique players: {df["name"].nunique()}')
    
    # Player minutes distribution
    if 'minutesPlayed' in df.columns:
        no_min = (df['minutesPlayed'] == 0).sum()
        low_min = ((df['minutesPlayed'] > 0) & (df['minutesPlayed'] <= 45)).sum()
        print(f'\nMinutes distribution:')
        print(f'  0 min (unused sub): {no_min:>6} ({no_min/df.shape[0]*100:5.1f}%)')
        print(f'  1-45 min (sub):     {low_min:>6} ({low_min/df.shape[0]*100:5.1f}%)')
    
    return df, df_master

sofascore_dynamic = {}
for s in ['2022-2023', '2023-2024', '2024-2025']:
    df, dfm = explore_sofascore_dynamic(s)
    if df is not None:
        sofascore_dynamic[s] = df

### 1.1 Missing Value Root-Cause Analysis

Let's understand WHY rating, ELO, and xG are missing.

In [ ]:
# Rating nulls: are they players with 0 minutes?
for s, df in sofascore_dynamic.items():
    null_rating = df['rating'].isna()
    if null_rating.any():
        zero_min_null = ((df['minutesPlayed'] == 0) & null_rating).sum()
        print(f'{s}: null rating where min==0: {zero_min_null}/{null_rating.sum()} ({zero_min_null/null_rating.sum()*100:.1f}%)')
        # Sample of non-zero min with null rating
        non_zero_null = df[null_rating & (df['minutesPlayed'] > 0)]
        if len(non_zero_null) > 0:
            print(f'  Non-zero min with null rating: {len(non_zero_null)} rows')
            print(f'  Sample: {non_zero_null[["name","teamName","minutesPlayed","match_date_str"]].head(3).to_string(index=False)}')

In [ ]:
# ELO nulls: which teams?
for s, df in sofascore_dynamic.items():
    if 'elo' not in df.columns:
        print(f'{s}: elo column missing entirely')
        continue
    null_elo = df['elo'].isna()
    if null_elo.any():
        teams_null = df.loc[null_elo, 'teamName'].value_counts()
        print(f'{s}: elo null for teams:')
        for t, c in teams_null.items():
            print(f'  {t}: {c} rows')

In [ ]:
# xG nulls: why are team_xg, team_xga, xg_difference 100% null in clean files?
# Check the master file for xG columns
for s in ['2022-2023', '2023-2024', '2024-2025']:
    master_path = BASE / 'Data' / s / 'sofascore_dynamic' / 'fixtureiq_dynamic_master.csv'
    if master_path.exists():
        dfm = pd.read_csv(master_path, nrows=5)
        xg_cols = [c for c in dfm.columns if 'xg' in c.lower() or 'xG' in c]
        print(f'{s} master xG columns: {xg_cols}')
        if 'home_xg' in dfm.columns:
            null_hxg = pd.read_csv(master_path, usecols=['home_xg']).isna().sum().iloc[0]
            total = pd.read_csv(master_path, usecols=['home_xg']).shape[0]
            print(f'  home_xg nulls: {null_hxg}/{total} ({null_hxg/total*100:.1f}%)')

### 1.2 Workload Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for idx, (s, df) in enumerate(sofascore_dynamic.items()):
    ax = axes[0, idx]
    if 'rest_days' in df.columns:
        df['rest_days'].clip(0, 30).hist(bins=30, ax=ax, alpha=0.7)
        ax.set_title(f'{s} — Rest Days Distribution')
        ax.set_xlabel('Rest days (clipped at 30)')
    
    ax2 = axes[1, idx]
    if 'acwr_ratio' in df.columns:
        df['acwr_ratio'].clip(0, 3).hist(bins=30, ax=ax2, alpha=0.7, color='orange')
        ax2.axvline(0.5, color='red', ls='--', alpha=0.5, label='danger low')
        ax2.axvline(1.5, color='red', ls='--', alpha=0.5, label='danger high')
        ax2.set_title(f'{s} — ACWR Distribution')
        ax2.set_xlabel('ACWR (clipped 0-3)')
        ax2.legend()

plt.tight_layout()
plt.savefig('../results/data_exploration_workload_dist.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Congestion flag rates across seasons
print('High Congestion (<=3 days rest) rates:')
for s, df in sofascore_dynamic.items():
    if 'high_congestion_flag' in df.columns:
        rate = df['high_congestion_flag'].mean() * 100
        print(f'  {s}: {rate:.1f}% of matches ({df["high_congestion_flag"].sum():,} of {df.shape[0]:,})')

---
## 2. FBref Per-Match Data

FBref covers specific UCL-participant PL teams per season. Let's check what's available.

In [ ]:
def explore_fbref_season(season_code):
    """Explore FBref data for a season."""
    base = BASE / 'Data' / season_code / 'fbref'
    if not base.exists():
        print(f'{season_code}: no FBref directory')
        return {}
    
    teams = {}
    for team_dir in sorted(base.iterdir()):
        if not team_dir.is_dir():
            continue
        
        # Count match reports
        mr_dir = team_dir / 'match_reports'
        n_matches = len([x for x in mr_dir.iterdir() if x.is_dir()]) if mr_dir.exists() else 0
        
        # Player stats files
        ps_dir = team_dir / 'player_stats'
        n_ps_files = len(list(ps_dir.glob('*.csv'))) if ps_dir.exists() else 0
        
        teams[team_dir.name] = {'match_reports': n_matches, 'player_stats_files': n_ps_files}
    
    return teams

for s in ['2022-2023', '2023-2024', '2024-2025']:
    teams = explore_fbref_season(s)
    if teams:
        print(f'=== {s} FBref ===')
        for t, info in teams.items():
            print(f'  {t}: {info["match_reports"]} matches, {info["player_stats_files"]} stat files')

In [ ]:
# Sample FBref match report columns & completeness
import glob
fbref_samples = {}
for s in ['2022-2023', '2023-2024', '2024-2025']:
    base = BASE / 'Data' / s / 'fbref'
    if not base.exists():
        continue
    for team_dir in sorted(base.iterdir()):
        if not team_dir.is_dir():
            continue
        mr_dir = team_dir / 'match_reports'
        if not mr_dir.exists():
            continue
        # Find first match report
        matches = sorted(mr_dir.iterdir())
        if not matches:
            continue
        for m in matches[:1]:  # just first match
            ps_file = m / f'{m.name}_player_stats.csv'
            if ps_file.exists():
                df = pd.read_csv(ps_file)
                fbref_samples[f'{s}/{team_dir.name}'] = {
                    'rows': df.shape[0],
                    'cols': df.shape[1],
                    'col_names': list(df.columns),
                    'sample': df.head(2).to_dict('records'),
                    'nulls': df.isnull().sum()[df.isnull().sum() > 0].to_dict()
                }
                break

for k, v in list(fbref_samples.items())[:3]:
    print(f'\n=== {k} ===')
    print(f'Shape: {v["rows"]} x {v["cols"]}')
    print(f'Columns: {v["col_names"]}')
    if v['nulls']:
        print(f'Nulls: {v["nulls"]}')

### 2.1 FBref Data Completeness

Let's verify that the `load_fbref_data()` function in the training pipeline correctly loads all available FBref data.

In [ ]:
# Simulate what the training pipeline loads
def load_fbref_simulated():
    """Replicate the load_fbref_data logic from train_fatigue_model.py"""
    all_rows = []
    for season_str in ['2022-2023', '2023-2024', '2024-2025']:
        season_path = BASE / 'Data' / season_str / 'fbref'
        if season_path.exists():
            for team_dir in sorted(season_path.iterdir()):
                if not team_dir.is_dir():
                    continue
                mr_dir = team_dir / 'match_reports'
                if not mr_dir.exists():
                    continue
                for match_folder in sorted(mr_dir.iterdir()):
                    if not match_folder.is_dir():
                        continue
                    ps_file = match_folder / f'{match_folder.name}_player_stats.csv'
                    if ps_file.exists():
                        all_rows.append({'season': season_str, 'team_dir': team_dir.name, 'match': match_folder.name, 'file': str(ps_file)})
    return pd.DataFrame(all_rows)

fbref_inventory = load_fbref_simulated()
print(f'FBref total match reports found: {fbref_inventory.shape[0]}')
print(f'\nBreakdown by season:')
print(fbref_inventory.groupby('season')['match'].count().to_string())
print(f'\nBreakdown by team:')
print(fbref_inventory.groupby('team_dir')['match'].count().to_string())

---
## 3. SofaScore Season Aggregate Data

These are season-level player aggregate stats with 115+ columns (all passing, defending, shooting, rating stats).

In [ ]:
def explore_sofascore_aggregates():
    """Check all SofaScore all_players files available."""
    pattern = str(BASE / 'Data' / '*' / 'sofascore' / '*' / '*_all_players.csv')
    files = sorted(glob.glob(pattern))
    
    print(f'Found {len(files)} aggregate player files:')
    info = []
    for f in files:
        df = pd.read_csv(f)
        parts = Path(f).relative_to(BASE).parts
        season = parts[1]
        comp = parts[3]
        info.append({'file': f, 'season': season, 'comp': comp, 'players': df.shape[0], 'cols': df.shape[1]})
        print(f'  {season}/{comp}: {df.shape[0]:>4} players, {df.shape[1]:>3} cols')
    return pd.DataFrame(info)

ss_agg = explore_sofascore_aggregates()

In [ ]:
# Check SofaScore aggregate nulls
f = ss_agg.iloc[0]['file']
df = pd.read_csv(f)
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(f'Nulls in {Path(f).name} ({df.shape[0]} players):')
if len(nulls) > 0:
    for c, v in nulls.items():
        print(f'  {c}: {v} ({v/df.shape[0]*100:.1f}%)')
else:
    print('  (none)')

---
## 4. Injury Data

The ground-truth target variable. Let's check coverage and quality.

In [ ]:
def explore_injuries():
    """Load and summarize injury data across all seasons."""
    all_inj = []
    for s in ['2022-2023', '2023-2024', '2024-2025']:
        inj_dir = BASE / 'Data' / s / 'injuries'
        if not inj_dir.exists():
            continue
        for f in sorted(inj_dir.iterdir()):
            if f.name.endswith('_injuries_days_out.csv'):
                df = pd.read_csv(f)
                df['season'] = s
                all_inj.append(df)
    
    if not all_inj:
        return pd.DataFrame()
    
    inj = pd.concat(all_inj, ignore_index=True)
    inj['fixture_date'] = pd.to_datetime(inj['fixture_date'], errors='coerce')
    inj['end_date'] = pd.to_datetime(inj['end_date'], errors='coerce')
    inj['days_out'] = pd.to_numeric(inj['days_out'], errors='coerce')
    
    return inj

injuries = explore_injuries()
print(f'Total injury records: {injuries.shape[0]:,}')
print(f'\nBy season:')
print(injuries.groupby('season').agg({'player_id': 'count', 'player_name': 'nunique', 'team_name': 'nunique'}).to_string())
print(f'\nDate range: {injuries["fixture_date"].min()} to {injuries["fixture_date"].max()}')
print(f'\nInjury types:')
print(injuries['injury_type'].value_counts().to_string())
print(f'\nDays out stats:')
print(injuries['days_out'].describe().to_string())

In [ ]:
# Genuine vs non-injury breakdown
GENUINE = {
    "Abdominal strain", "Achilles Tendon Injury", "Ankle Injury", "Arm Injury",
    "Back Injury", "Broken Leg", "Calf Injury", "Concussion", "Eye injury",
    "Face Injury", "Finger Injury", "Foot Injury", "Groin Injury",
    "Hamstring Injury", "Hand Injury", "Head Injury", "Heel Injury",
    "Hip Injury", "Illness", "Injury", "Knee Injury", "Knock",
    "Leg Injury", "Lower Back Injury", "Muscle Injury", "Pelvis Injury",
    "Shoulder Injury", "Thigh Injury", "Toe Injury", "Wrist Injury",
}
NON_INJURY = {
    "Inactive", "Suspended", "Red Card", "Yellow Cards",
    "Coach's decision", "Loan agreement", "Personal Reasons",
    "Rest", "Health problems", "Convalescence", "Lacking Match Fitness",
}

injuries['is_injury'] = injuries['reason'].isin(GENUINE)
injuries['is_non_injury'] = injuries['reason'].isin(NON_INJURY)

print(f'Genuine injuries: {injuries["is_injury"].sum():,} ({injuries["is_injury"].mean()*100:.1f}%)')
print(f'Non-injury (suspension/rest/etc): {injuries["is_non_injury"].sum():,} ({injuries["is_non_injury"].mean()*100:.1f}%)')
print(f'Unclassified: {(~injuries["is_injury"] & ~injuries["is_non_injury"]).sum():,}')

print(f'\nTop 15 reasons:')
print(injuries['reason'].value_counts().head(15).to_string())

In [ ]:
# Teams covered by injury data
print('Teams per season:')
for s in ['2022-2023', '2023-2024', '2024-2025']:
    subset = injuries[injuries['season'] == s]
    teams = sorted(subset['team_name'].unique())
    print(f'  {s}: {len(teams)} teams -> {teams}')

---
## 5. Context Data: Club Elo & Understat

These enrich the match data with team strength and expected goals.

In [ ]:
# Club Elo
elo = pd.read_csv(BASE / 'Data' / 'clubelo_understat' / 'fixtureiq_elo_master.csv')
print(f'ClubElo: {elo.shape[0]} entries, {elo["team"].nunique()} unique teams')
print(f'Date range: {elo["from"].min()} to {elo["to"].max()}')
print(f'\nPL teams in Elo:')
pl_teams = ['Arsenal','Aston Villa','Bournemouth','Brentford','Brighton','Chelsea','Crystal Palace',
            'Everton','Fulham','Ipswich','Leicester','Liverpool','Man City','Man United',
            'Newcastle','Nottingham Forest','Southampton','Tottenham','West Ham','Wolves']
for t in pl_teams:
    found = elo[elo['team'] == t].shape[0]
    if found == 0:
        # try variations
        for alt in [t+' FC', t.replace(' ',''), t.replace(' ','_')]:
            found = elo[elo['team'] == alt].shape[0]
            if found > 0:
                print(f'  {t} -> found as "{alt}" ({found} entries)')
                break
        else:
            print(f'  {t}: NOT FOUND in Elo')

In [ ]:
# Understat
un = pd.read_csv(BASE / 'Data' / 'clubelo_understat' / 'fixtureiq_understat_master.csv')
print(f'Understat: {un.shape[0]} matches, seasons={sorted(un["season"].unique())}')
print(f'Date range: {un["date"].min()} to {un["date"].max()}')
print(f'\nTeam name examples:')
print(f'  Home teams: {sorted(un["home_team"].unique())[:15]}')
print(f'  Away teams: {sorted(un["away_team"].unique())[:15]}')

In [ ]:
# Check why Understat xG fails to merge: name mismatch
# The dynamic pipeline uses TEAM_NAME_MAP but let's verify
TEAM_NAME_MAP = {
    'Arsenal': 'Arsenal', 'Aston Villa': 'Aston Villa',
    'Liverpool FC': 'Liverpool', 'Manchester City': 'Manchester City',
    'Manchester United': 'Manchester United', 'Newcastle United': 'Newcastle United',
    'Chelsea': 'Chelsea', 'Tottenham Hotspur': 'Tottenham Hotspur',
    'Tottenham': 'Tottenham Hotspur', 'Liverpool': 'Liverpool',
    'Brighton & Hove Albion': 'Brighton', 'Brighton': 'Brighton',
    'West Ham United': 'West Ham United', 'West Ham': 'West Ham United',
    'Wolverhampton': 'Wolverhampton', 'Wolves': 'Wolverhampton',
    'Leicester City': 'Leicester City', 'Leicester': 'Leicester City',
    'Ipswich Town': 'Ipswich Town', 'Southampton': 'Southampton',
    'Bournemouth': 'Bournemouth', 'Brentford': 'Brentford',
    'Crystal Palace': 'Crystal Palace', 'Everton': 'Everton',
    'Fulham': 'Fulham', 'Nottingham Forest': 'Nottingham Forest',
    "Nott'ham Forest": 'Nottingham Forest',
}

# Check: Understat team names that DON'T have a reverse mapping
dyn_team = 'Liverpool FC'
canon = TEAM_NAME_MAP.get(dyn_team, dyn_team)
print(f'Dynamic "{dyn_team}" -> canonical "{canon}"')
print(f'Understat has "{canon}": {"Liverpool" in un["home_team"].values}')
print()

# Check all dynamic PL teams vs Understat names
dyn = sofascore_dynamic['2022-2023']
dyn_teams = set(dyn['teamName'].unique())
un_teams = set(un['home_team'].unique()) | set(un['away_team'].unique())
print('Dynamic teams lacking Understat match:')
for t in sorted(dyn_teams):
    canon = TEAM_NAME_MAP.get(t, t)
    if canon not in un_teams and t not in un_teams:
        print(f'  {t} -> canon={canon} (Understat has: {[x for x in un_teams if x.lower()[:4]==canon.lower()[:4]]})')

---
## 6. Name Matching Across Sources

A critical issue: merging across SofaScore, FBref, and Injury data requires aligning player names.

In [ ]:
# Compare player name formats across sources
dyn_players = set()
for s, df in sofascore_dynamic.items():
    if 'name' in df.columns:
        dyn_players.update(df['name'].unique())

# FBref names from match reports
fbref_players = set()
for s in ['2022-2023', '2023-2024', '2024-2025']:
    base = BASE / 'Data' / s / 'fbref'
    if not base.exists():
        continue
    for team_dir in base.iterdir():
        if not team_dir.is_dir():
            continue
        mr_dir = team_dir / 'match_reports'
        if not mr_dir.exists():
            continue
        for match_folder in mr_dir.iterdir():
            if not match_folder.is_dir():
                continue
            ps_file = match_folder / f'{match_folder.name}_player_stats.csv'
            if ps_file.exists():
                df = pd.read_csv(ps_file)
                if 'Player' in df.columns:
                    fbref_players.update(df['Player'].unique())

# Injury names
injury_players = set(injuries['player_name'].unique())

print(f'SofaScore Dynamic players: {len(dyn_players):,}')
print(f'FBref players: {len(fbref_players):,}')
print(f'Injury players: {len(injury_players):,}')
print()

# Show some examples of each format
print('5 SofaScore names:')
for n in sorted(dyn_players)[:5]:
    print(f'  "{n}"')
print('\n5 FBref names:')
for n in sorted(fbref_players)[:5]:
    print(f'  "{n}"')
print('\n5 Injury names:')
for n in sorted(injury_players)[:5]:
    print(f'  "{n}"')

In [ ]:
# Test the current last_name matching strategy
injuries['last_name'] = injuries['player_name'].str.split().str[-1].str.lower().str.strip()
dyn_players_df = pd.DataFrame({'name': list(dyn_players)})
dyn_players_df['last_name'] = dyn_players_df['name'].str.split().str[-1].str.lower().str.strip()

# How many injury player last names find a match in dynamic data?
matched_names = injuries['last_name'].isin(dyn_players_df['last_name']).sum()
print(f'Injury last_names matched in dynamic data: {matched_names}/{len(injuries)} ({matched_names/len(injuries)*100:.1f}%)')

# Unmatched examples
unmatched = injuries[~injuries['last_name'].isin(dyn_players_df['last_name'])]['player_name'].unique()
print(f'\nUnmatched injury player names ({len(unmatched)}):')
for n in sorted(unmatched)[:10]:
    print(f'  "{n}" -> last_name="{n.split()[-1].lower()}"')

In [ ]:
# Check the "Martin Odegaard" / "Martin Ødegaard" problem
target = 'odegaard'
dyn_matches = dyn_players_df[dyn_players_df['last_name'].str.contains(target, na=False)]
inj_matches = injuries[injuries['last_name'].str.contains(target, na=False)]
fbref_matches = [p for p in fbref_players if target in p.lower()]

print(f'Dynamic match for "{target}":')
for n in dyn_matches['name'].unique():
    print(f'  "{n}"')
print(f'\nFBref matches:')
for n in fbref_matches:
    print(f'  "{n}"')
print(f'\nInjury matches:')
for n in inj_matches['player_name'].unique():
    print(f'  "{n}"')

---
## 7. Cross-Source Unification Feasibility

Let's compare what a unified dataset would look like.

In [ ]:
# Build a comparison of overlapping columns
all_cols = {}
for s, df in sofascore_dynamic.items():
    all_cols[f'SofaScore_Dynamic_{s}'] = set(df.columns)

# Add clean file columns
print('Column overlap between seasons (SofaScore Dynamic clean files):')
seasons = list(sofascore_dynamic.keys())
for i in range(len(seasons)):
    for j in range(i+1, len(seasons)):
        s1, s2 = seasons[i], seasons[j]
        c1 = set(sofascore_dynamic[s1].columns)
        c2 = set(sofascore_dynamic[s2].columns)
        common = c1 & c2
        only1 = c1 - c2
        only2 = c2 - c1
        print(f'\n{s1} vs {s2}:')
        print(f'  Common: {len(common)} cols')
        if only1: print(f'  Only in {s1}: {sorted(only1)}')
        if only2: print(f'  Only in {s2}: {sorted(only2)}')

In [ ]:
# Build a unified data availability matrix
print('=== Data Availability Matrix ===')
print(f'{"Data Source":<45} {"22-23":<10} {"23-24":<10} {"24-25":<10}')
print('-' * 75)

sources = [
    ('SofaScore Dynamic (clean)', 6500, 13142, 6726),
    ('SofaScore Dynamic (master)', 7305, 14758, 7588),
    ('FBref per-match (UCL teams)', '4 teams', '4 teams', '4 teams'),
    ('SofaScore Aggregates (PL)', '1 file', '1 file', '1 file'),
    ('SofaScore Aggregates (UCL)', '1 file', '1 file', '1 file'),
    ('Injuries (all PL teams)', '2774 rec', '3601 rec', '3154 rec'),
    ('Position Profiles (PL)', '6 files', '6 files', '6 files'),
    ('Position Profiles (UCL)', '6 files', '6 files', '6 files'),
    ('Average Positions', '-', '-', '75+ files'),
]

for name, s1, s2, s3 in sources:
    print(f'{name:<45} {str(s1):<10} {str(s2):<10} {str(s3):<10}')

print()
print('Context data (shared across all seasons):')
print('  ClubElo: 631 teams, date-windowed ratings')
print('  Understat: 1520 PL matches, seasons 2020-21 to 2023-24')

---
## 8. Key Findings & Recommendations

### Data Quality Issues Found:

1. **Null SofaScore ratings (~24%)** — Almost entirely players with 0 minutes played (unused substitutes). Easy to handle.
2. **Null ELO (~10%)** — Teams that aren't in ClubElo (some non-PL teams from UCL matches like "AFC Ajax", "GNK Dinamo Zagreb"). Can impute with league average.
3. **Null team_xg (100% in clean files)** — The Understat merge in `fixtureiq_dynamic_elite.py` failed because:
   - Team name mismatch between SofaScore dynamic names and Understat names
   - The `TEAM_NAME_MAP` covers PL teams but doesn't fully align with Understat naming
   - The clean file export may drop these columns
4. **23-24 clean file missing ELO and xG columns** entirely (only 17 vs 19 cols) — Original pipeline didn't include them.

### Name Matching Issues:
5. **SofaScore uses full names** ("Aaron Cresswell") vs **Injury uses abbreviated first names** ("A. Cresswell"). Current `last_name`-only match works but could miss cases.
6. **Diacritics**: "Odegaard" vs "Ødegaard" — the last_name match works but direct name match would fail.
7. **FBref names** use full names but sometimes with different formatting.

### Coverage Gaps:
8. **Non-UCL teams only available in dynamic data for 24-25** — 22-23 and 23-24 dynamic data only covers UCL teams.
9. **Understat data ends at 2023-24** — No xG context for 24-25 matches.
10. **Average positions only for 24-25** — Can't use for multi-season model.

In [ ]:
# Summary stats for final unified view
total_obs = sum(df.shape[0] for df in sofascore_dynamic.values())
total_players = len(set().union(*[set(df['name'].unique()) for df in sofascore_dynamic.values()]))
total_injuries = injuries.shape[0]
total_genuine = injuries['is_injury'].sum()

print('=' * 60)
print('UNIFIED DATA SUMMARY')
print('=' * 60)
print(f'Total player-match observations: {total_obs:,}')
print(f'Total unique players: {total_players:,}')
print(f'Total injury records: {total_injuries:,}')
print(f'  Genuine injuries: {total_genuine:,} ({total_genuine/total_injuries*100:.1f}%)')
print()

# Per-season breakdown
print(f'{"Season":<12} {"Matches":<10} {"Players":<10} {"Teams":<10} {"Injuries":<10}')
print('-' * 52)
for s in ['2022-2023', '2023-2024', '2024-2025']:
    df = sofascore_dynamic.get(s)
    inj_season = injuries[injuries['season'] == s]
    if df is not None:
        print(f'{s:<12} {df.shape[0]:<10,} {df["name"].nunique():<10} {df["teamName"].nunique():<10} {inj_season.shape[0]:<10,}')

In [ ]:
print('\nDone. All data sources explored.')